# 🧠 GALILEO BRAIN V5 — Fine-tuning Qwen3-4B (25K Dataset)
## ELAB Tutor — Andrea Marro — 2026

### Dataset Factory v5: 22,480 training + 191 eval examples
### 7 intents: action, circuit, code, tutor, teacher, vision, navigation
### Output: JSON routing (intent, entities, actions, needs_llm, response)

**Istruzioni:**
1. Esegui Cell 1 (installazione)
2. **Riavvia il runtime** (Runtime → Riavvia)
3. Esegui tutte le celle da 2 in poi
4. Carica i 2 dataset quando richiesto (training + eval)
5. Attendi il training (~2h su T4, ~45min su A100)
6. Scarica il modello GGUF

In [ ]:
# Cell 1: Installazione Unsloth + dipendenzeimport sysif "google.colab" in sys.modules:    !pip install --no-deps trl peft accelerate bitsandbytes    !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"    !pip install --no-deps xformers triton    !pip install transformers==4.56.2 trl==0.22.2 datasetsprint("✅ Installazione completata!")print("⚠️  ORA RIAVVIA IL RUNTIME: Runtime → Riavvia runtime")print("   Poi esegui dalla Cell 2 in poi")

In [ ]:
# Cell 2: Carica Modello + Configurazione LoRA
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length=2048,
    dtype=None,  # auto-detect
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,                    # LoRA rank (32 = buon bilanciamento qualità/velocità)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,           # 1:1 ratio (stabile per dataset grandi)
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print(f"✅ Modello caricato: Qwen3-4B-Instruct")
print(f"   LoRA rank=32, alpha=32, dropout=0.05")
print(f"   Parametri trainabili: {model.print_trainable_parameters()}")

In [ ]:
# Cell 3: Carica Dataset (Training + Eval separati)
from google.colab import files
import json

print("📂 Carica 2 file:")
print("   1. galileo-brain-v5-extreme-25k.jsonl  (training ~22K)")
print("   2. galileo-brain-v5-eval-200.jsonl     (eval ~191)")
print()
uploaded = files.upload()

# Identifica i file
train_file = None
eval_file = None
for name in uploaded.keys():
    if "eval" in name:
        eval_file = name
    else:
        train_file = name

print(f"\n✅ Training: {train_file}")
print(f"✅ Eval:     {eval_file}")

# Conta e verifica
for label, fpath in [("Training", train_file), ("Eval", eval_file)]:
    if fpath:
        with open(fpath) as f:
            lines = f.readlines()
        sample = json.loads(lines[0])
        output = json.loads(sample["messages"][2]["content"])
        print(f"\n📋 {label}: {len(lines)} esempi")
        print(f"   User: {sample['messages'][1]['content'][:80]}...")
        print(f"   Intent: {output.get('intent')} | needs_llm: {output.get('needs_llm')}")
        print(f"   Actions: {output.get('actions', [])[:2]}")

In [ ]:
# Cell 4: Prepara Dataset per Training
from datasets import Dataset
import json

# Carica training examples
train_examples = []
with open(train_file, 'r') as f:
    for line in f:
        train_examples.append(json.loads(line.strip()))

# Carica eval examples (dataset separato con seed diverso)
eval_examples = []
if eval_file:
    with open(eval_file, 'r') as f:
        for line in f:
            eval_examples.append(json.loads(line.strip()))

train_dataset = Dataset.from_list(train_examples)
eval_dataset = Dataset.from_list(eval_examples) if eval_examples else None

print(f"✅ Dataset preparato:")
print(f"   Training: {len(train_dataset)} esempi")
print(f"   Evaluation: {len(eval_dataset) if eval_dataset else 0} esempi")

# Formatta con chat template
def formatting_func(examples):
    convos = examples["messages"]
    texts = []
    for convo in convos:
        text = tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}

train_dataset = train_dataset.map(formatting_func, batched=True)
if eval_dataset:
    eval_dataset = eval_dataset.map(formatting_func, batched=True)

# Statistiche intents
intent_counts = {}
for ex in train_examples:
    try:
        out = json.loads(ex["messages"][2]["content"])
        intent = out.get("intent", "?")
        intent_counts[intent] = intent_counts.get(intent, 0) + 1
    except:
        pass

print(f"\n📊 Distribuzione intents nel training set:")
for intent, count in sorted(intent_counts.items(), key=lambda x: -x[1]):
    pct = 100 * count / len(train_examples)
    bar = "█" * int(pct / 2)
    print(f"   {intent:15s} {count:5d} ({pct:4.1f}%) {bar}")

print(f"\n📋 Esempio formattato:")
print(train_dataset[0]["text"][:400] + "...")

In [ ]:
# Cell 5: Configura SFT Trainer
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

# Con ~22K esempi e batch 16: ~1,400 steps/epoch × 3 epochs = ~4,200 steps
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    dataset_num_proc=2,
    args=SFTConfig(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,   # Effective batch = 16
        warmup_steps=100,                # Più warmup per dataset grande
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="outputs",
        eval_strategy="steps",
        eval_steps=200,
        save_strategy="steps",
        save_steps=500,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
    ),
)

# Train SOLO su risposte dell'assistente (non su system/user)
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

print("✅ Trainer configurato:")
print("   Batch: 4 × 4 = 16 effective")
print("   Epochs: 3")
print("   LR: 2e-4 con cosine scheduler")
print("   Warmup: 100 steps")
print("   Eval ogni 200 steps, save ogni 500")
print("   Loss SOLO su risposte assistant")

In [ ]:
# Cell 6: TRAINING 🚀import timeprint("🚀 Avvio training...")print("   Tempo stimato: 30-45 minuti su T4")print("   Target: loss finale < 0.1")print("=" * 50)start = time.time()stats = trainer.train()elapsed = time.time() - startprint(f"\n✅ Training completato in {elapsed/60:.1f} minuti!")print(f"   Loss finale: {stats.training_loss:.4f}")print(f"   Steps totali: {stats.global_step}")

In [ ]:
# Cell 7: Test Inferenza — Verifica tutti i 7 intent
FastLanguageModel.for_inference(model)

test_cases = [
    # ACTION — telegraphic
    {"message": "Fai partire la simulazione!", "expected_intent": "action"},
    {"message": "ferma tutto", "expected_intent": "action"},
    {"message": "compila il codice", "expected_intent": "action"},
    # CIRCUIT — component placement
    {"message": "Metti un LED rosso sulla breadboard", "expected_intent": "circuit"},
    {"message": "aggiungi una resistenza da 220 ohm", "expected_intent": "circuit"},
    # CODE
    {"message": "Scrivi il codice per far lampeggiare il LED", "expected_intent": "code"},
    {"message": "apri Scratch", "expected_intent": "code"},
    # TUTOR — educational
    {"message": "Cos'è la legge di Ohm?", "expected_intent": "tutor"},
    {"message": "perché il LED ha bisogno di una resistenza?", "expected_intent": "tutor"},
    # NAVIGATION
    {"message": "Apri il volume 2", "expected_intent": "navigation"},
    {"message": "vai all'esperimento successivo", "expected_intent": "navigation"},
    # VISION
    {"message": "guarda il mio circuito e dimmi se è giusto", "expected_intent": "vision"},
    # TEACHER — didactic
    {"message": "Come spiego la legge di Ohm a una classe di terza media?", "expected_intent": "teacher"},
    {"message": "Non capisco niente di elettronica, da dove comincio?", "expected_intent": "teacher"},
    # HARD: corrupted input
    {"message": "meti un led roso", "expected_intent": "circuit"},
    # HARD: profanity
    {"message": "ma che cazzo non funziona il LED", "expected_intent": "tutor"},
    # HARD: emoji
    {"message": "▶️", "expected_intent": "action"},
    # HARD: out of scope
    {"message": "chi ha vinto la Champions League?", "expected_intent": "action"},
]

# Leggi il system prompt dal primo esempio del dataset
with open(train_file) as f:
    sys_prompt = json.loads(f.readline())["messages"][0]["content"]

print(f"🧪 Test Inferenza — {len(test_cases)} casi")
print("=" * 60)

passed = 0
for i, tc in enumerate(test_cases):
    messages = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": tc["message"]}
    ]

    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs, max_new_tokens=512, temperature=0.1,
        top_p=0.95, do_sample=True,
    )

    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)

    try:
        # Handle potential thinking tokens or multi-line
        clean = response.strip()
        if clean.startswith("{"):
            json_str = clean.split("\n")[0] if "\n" in clean else clean
        else:
            # Find first { in response
            idx = clean.find("{")
            json_str = clean[idx:] if idx >= 0 else clean
        parsed = json.loads(json_str)
        intent_ok = parsed.get("intent") == tc["expected_intent"]
        status = "✅" if intent_ok else "❌"
        if intent_ok:
            passed += 1
        print(f"{status} Test {i+1}: '{tc['message'][:45]}...'")
        print(f"   Expected: {tc['expected_intent']} | Got: {parsed.get('intent')}")
        if parsed.get("actions"):
            print(f"   Actions: {parsed['actions'][:2]}")
    except:
        print(f"❌ Test {i+1}: JSON parse error")
        print(f"   Raw: {response[:150]}...")

print(f"\n{'='*60}")
print(f"Score: {passed}/{len(test_cases)} ({100*passed/len(test_cases):.0f}%)")
print(f"Target: ≥ 85%")

In [ ]:
# Cell 8: Evaluation Suite su Eval Set (191 esempi)
import json

# Usa il vero eval set (seed diverso, no data leakage)
eval_examples_raw = []
with open(eval_file, 'r') as f:
    for line in f:
        eval_examples_raw.append(json.loads(line.strip()))

# Leggi system prompt
with open(train_file) as f:
    sys_prompt = json.loads(f.readline())["messages"][0]["content"]

correct_intent = 0
correct_actions = 0
correct_needs_llm = 0
parse_errors = 0
total = len(eval_examples_raw)

intent_matrix = {}  # {expected: {predicted: count}}

print(f"🧪 Evaluation su {total} esempi (eval holdout set)")
print("=" * 50)

for example in eval_examples_raw:
    expected = json.loads(example["messages"][2]["content"])
    messages = [
        {"role": "system", "content": sys_prompt},
        example["messages"][1],  # user message
    ]

    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs, max_new_tokens=512, temperature=0.1, do_sample=True
    )

    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)

    try:
        clean = response.strip()
        idx = clean.find("{")
        json_str = clean[idx:] if idx >= 0 else clean
        # Handle multi-line JSON
        if "\n" in json_str:
            # Try full string first, then first line
            try:
                parsed = json.loads(json_str)
            except:
                parsed = json.loads(json_str.split("\n")[0])
        else:
            parsed = json.loads(json_str)

        exp_intent = expected["intent"]
        pred_intent = parsed.get("intent", "?")

        # Confusion matrix
        if exp_intent not in intent_matrix:
            intent_matrix[exp_intent] = {}
        intent_matrix[exp_intent][pred_intent] = intent_matrix[exp_intent].get(pred_intent, 0) + 1

        if pred_intent == exp_intent:
            correct_intent += 1

        # Actions match
        exp_actions = set(expected.get("actions", []))
        pred_actions = set(parsed.get("actions", []))
        if exp_actions == pred_actions:
            correct_actions += 1

        # needs_llm match
        if parsed.get("needs_llm") == expected.get("needs_llm"):
            correct_needs_llm += 1
    except:
        parse_errors += 1

valid = total - parse_errors
print(f"\n📊 Risultati ({valid} validi / {total} totali):")
print(f"   Intent accuracy:    {100*correct_intent/total:.1f}%")
print(f"   Actions accuracy:   {100*correct_actions/total:.1f}%")
print(f"   needs_llm accuracy: {100*correct_needs_llm/total:.1f}%")
print(f"   Parse errors:       {parse_errors}")
print(f"\n   Target: intent ≥ 90%")

# Confusion matrix
print(f"\n📊 Confusion Matrix:")
all_intents = sorted(set(list(intent_matrix.keys()) +
    [p for row in intent_matrix.values() for p in row.keys()]))
header = f"{'':>12}" + "".join(f"{i[:6]:>8}" for i in all_intents)
print(header)
for exp in all_intents:
    row = intent_matrix.get(exp, {})
    cells = "".join(f"{row.get(pred, 0):>8}" for pred in all_intents)
    print(f"{exp[:12]:>12}{cells}")

In [ ]:
# Cell 9: Salva Modello + Export GGUF
print("💾 Salvataggio LoRA adapter...")
model.save_pretrained("galileo-brain-v5-lora")
tokenizer.save_pretrained("galileo-brain-v5-lora")
print("✅ LoRA adapter salvato")

print("\n📦 Export GGUF q4_k_m per Ollama...")
model.save_pretrained_gguf(
    "galileo-brain-v5-gguf",
    tokenizer,
    quantization_method="q4_k_m"
)
print("✅ GGUF esportato!")

# Verifica file
import os
gguf_files = [f for f in os.listdir("galileo-brain-v5-gguf") if f.endswith('.gguf')]
for f in gguf_files:
    size_mb = os.path.getsize(f"galileo-brain-v5-gguf/{f}") / (1024*1024)
    print(f"   {f}: {size_mb:.1f} MB")

In [ ]:
# Cell 10: Scarica il modello + Crea Modelfile per Ollama
from google.colab import files
import os

gguf_dir = "galileo-brain-v5-gguf"
gguf_files = [f for f in os.listdir(gguf_dir) if f.endswith('.gguf')]

if gguf_files:
    gguf_path = os.path.join(gguf_dir, gguf_files[0])
    gguf_name = gguf_files[0]
    print(f"📥 Download: {gguf_name}")
    files.download(gguf_path)
    print("✅ Download avviato!")

    # Crea e scarica il Modelfile per Ollama
    modelfile_content = f"""FROM ./{gguf_name}

PARAMETER temperature 0.1
PARAMETER top_p 0.95
PARAMETER num_predict 512
PARAMETER stop "<|im_end|>"

SYSTEM \"\"\"Sei il BRAIN (cervello rapido) di ELAB Tutor, piattaforma educativa di elettronica per bambini 8-14 anni e docenti.
Il tuo UNICO compito: analizzare il messaggio e restituire un JSON di routing.
NON sei un chatbot. NON conversi. Output SOLO JSON valido.
Intent possibili: action, circuit, code, tutor, vision, navigation, teacher.
Rispondi SEMPRE e SOLO con JSON: {{"intent":"...", "entities":[...], "actions":[...], "needs_llm":true/false, "response":"...", "llm_hint":"..."}}\"\"\"
"""

    with open("Modelfile", "w") as f:
        f.write(modelfile_content)
    files.download("Modelfile")

    print("\n📋 Per usare con Ollama:")
    print(f"   1. Copia {gguf_name} + Modelfile nella stessa cartella")
    print("   2. ollama create galileo-brain -f Modelfile")
    print("   3. ollama run galileo-brain")
    print("\n   Il modello risponde SOLO JSON — perfetto per routing!")
else:
    print("❌ Nessun file GGUF trovato")